# MLflow solves the following three problems:

* `Experiment Tracking` - Every model run gets logged automatically. Parameters, metrics, artifacts. Timestamped. Named. Searchable. You never lose a result again.

* `Model Registry` - Your best model gets registered with a version number. Version 1 goes to Staging. After validation it gets promoted to Production. Your serving layer always knows which version is live.

* `Reproducibility` - Any run can be re-executed exactly. Same parameters, same data version, same result. That's the production guarantee.

# Phase 4 - MLflow Experiment Tracking

**Prerequisites:**
- MLflow UI running at http://127.0.0.1:5000
- model_table_enriched.csv in outputs/ folder

**Run in terminal before running the cells in the notebook**
```bash
cd ENTERPRISE_HEALTHCARE_ML
mlflow ui
```

In [7]:
import mlflow
import os
# Point MLflow to project root
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
print("MLflow Version : ", mlflow.__version__)
print("Tracking URI : ", mlflow.get_tracking_uri())

MLflow Version :  3.14.0
Tracking URI :  sqlite:///../mlflow.db


In [8]:
# Set the experiment
mlflow.set_experiment("healthcare-risk-classification")
print("Experiment Created ✔")

Experiment Created ✔


# Loading the saved model

In [ ]:
import os
import pandas as pd
import joblib
import warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import (accuracy_score, f1_score, recall_score)

In [12]:
# Load the saved models

risk_rf_model = joblib.load("../models/risk_model.joblib")
claim_rf_model = joblib.load("../models/claim_model.joblib")
print("Models Loaded ✔")
print("risk model : ", type(risk_rf_model))
print("claim model : ", type(claim_rf_model))

Models Loaded ✔
risk model :  <class 'sklearn.pipeline.Pipeline'>
claim model :  <class 'sklearn.pipeline.Pipeline'>


In [15]:
# Load the dataset

df = pd.read_csv("../outputs/model_table_enriched.csv", parse_dates = ["registration_date", "visit_date", "billing_date"])
print("Data Loaded ✔")
print("Shape of Data : ", df.shape)
df.head()

Data Loaded ✔
Shape of Data :  (25000, 30)


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,...,risk_numeric,claim_numeric,is_rejected,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag
0,2,15,F,Mumbai,CareOne,0,2025-12-27,8,2026-01-01,General,...,0,2,1,5,4,21.120000,0.256876,1,3,0
1,12,3,M,Bangalore,CareOne,0,2025-08-13,65,2026-01-01,ICU,...,2,2,1,141,8,23.750000,0.256876,1,3,1
2,129,44,M,Pune,MediCareX,1,2025-07-20,651,2026-01-01,ICU,...,2,1,0,165,3,32.460000,0.242556,1,3,1
3,133,47,F,Delhi,CareOne,1,2025-11-02,670,2026-01-01,General,...,1,0,0,60,3,30.056667,0.256876,1,3,0
4,139,14,F,Chennai,SecureLife,1,2025-02-05,706,2026-01-01,Cardiology,...,1,0,0,330,9,29.030000,0.157496,1,3,1


In [17]:
# Load feature schema
import json
with open("../outputs/feature_schema.json", "r") as f:
    schema = json.load(f)

risk_features = schema["risk_model_features"]
claim_features = schema["claim_model_features"]

risk_target = schema["risk_target"]
claim_target = schema["claim_target"]

print("Schema Loaded ✔")
print(f"Risk features : {len(risk_features)}")
print(f"Claim features : {len(claim_features)}")

Schema Loaded ✔
Risk features : 13
Claim features : 17


In [19]:
# Create separate datasets

risk_df = df.copy()
claim_df = df.copy()

In [20]:
# Time based split
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)
split_idx = int(len(risk_df)*0.8)

risk_train = risk_df.iloc[:split_idx].copy()
risk_test = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train[risk_features]
X_test_risk = risk_test[risk_features]

y_train_risk = risk_train[risk_target]
y_test_risk = risk_test[risk_target]

print(f"Train Shape: {X_train_risk.shape}")
print(f"Test Shape: {X_test_risk.shape}")
print(f"Train Period: {risk_train['visit_date'].min().date()} --> {risk_train['visit_date'].max().date()}")
print(f"Test Period: {risk_test['visit_date'].min().date()} --> {risk_test['visit_date'].max().date()}")

Train Shape: (20000, 13)
Test Shape: (5000, 13)
Train Period: 2025-01-21 --> 2026-01-02
Test Period: 2026-01-02 --> 2026-01-20


In [ ]:
# Time based split for claim

claim_df = claim_df.sort_values("billing_date").reset_index(drop=True)
split_idx = int(len(claim_df)*0.8)

claim_train = claim_df.iloc[: split_idx].copy()
claim_test = claim_df.iloc[split_idx : ].copy()

X_train_claim = claim_train[claim_features]
X_test_claim = claim_test[claim_features]

y_train_claim = claim_train[claim_target]
y_test_claim = claim_test[claim_target]

print(f"Train Shape: {X_train_claim.shape}")
print(f"Test Shape: {X_test_claim.shape}")
print(f"Train period : {claim_train['billing_date'].min().date()} -> {claim_train['billing_date'].max().date()}")
print(f"Test period : {claim_test['billing_date'].min().date()} -> {claim_test['billing_date'].max().date()}")

Train Shape: (20000, 17)
Test Shape: (5000, 17)
Train period : 2025-01-28 -> 2026-01-17
Test period : 2026-01-17 -> 2026-01-20


In [26]:
# Log Risk model

with mlflow.start_run(run_name = "RandomForest-risk") as run:
    mlflow.log_params({
        "model" : "RandomForestClassifier",
        "n_estimators" : 200,
        "max_depth" : 8,
        "min_samples_split" : 20,
        "min_samples_leaf" : 10,
        "class_weight" : "balanced_subsample",
        "evaluation_data" : "risk_test_only",
        "split_strategy" : "time_based_80_20",
        "split_column" : "visit_date"
    })

    pred_risk = risk_rf_model.predict(X_test_risk)

    # Metrics
    acc_risk = accuracy_score(y_test_risk, pred_risk)
    f1_risk = f1_score(y_test_risk, pred_risk, average="weighted")
    high_recall = recall_score(y_test_risk, pred_risk, labels=["High"], average=None)[0]

    # Logging the metrics
    mlflow.log_metric("accuracy", acc_risk)
    mlflow.log_metric("weighted_f1", f1_risk)
    mlflow.log_metric("high_risk_recall", high_recall)

    # Logging the Model
    mlflow.sklearn.log_model(
        sk_model = risk_rf_model,
        name = "model",
        skops_trusted_types=["numpy.dtype"]
    )

    #Fetching the run id
    risk_run_id = run.info.run_id

    print("RandomForest Risk Model logged ✓")
    print(f"  Accuracy         : {acc_risk:.4f}")
    print(f"  Weighted F1      : {f1_risk:.4f}")
    print(f"  High Risk Recall : {high_recall:.4f}")
    print(f"  Run ID           : {risk_run_id}")


2026/07/22 15:01:38 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


RandomForest Risk Model logged ✓
  Accuracy         : 0.9308
  Weighted F1      : 0.9307
  High Risk Recall : 0.9332
  Run ID           : 5ce2933f9346427388a5b210020a9396


In [27]:
# Log Claim Model
with mlflow.start_run(run_name="RandomForest-Claim") as run:

    mlflow.log_params({
        "model": "RandomForestClassifier",
        "n_estimators": 250,
        "max_depth": 14,
        "min_samples_split": 8,
        "class_weight": "balanced",
        "evaluation_data": "claim_test_only",
        "split_strategy": "time_based_80_20",
        "split_column": "billing_date"
    })

    pred_claim = claim_rf_model.predict(X_test_claim)

    acc_claim = accuracy_score(y_test_claim, pred_claim)
    f1_claim = f1_score(y_test_claim, pred_claim, average="weighted")
    rejected_recall = recall_score(
        y_test_claim, pred_claim, labels=["Rejected"], average=None
    )[0]

    mlflow.log_metric("accuracy", acc_claim)
    mlflow.log_metric("weighted_f1", f1_claim)
    mlflow.log_metric("rejected_recall", rejected_recall)

    mlflow.sklearn.log_model(
        sk_model=claim_rf_model,
        name="model",
        skops_trusted_types=["numpy.dtype"]
    )

    claim_run_id = run.info.run_id

    print("RandomForest Claim Model logged ✓")
    print(f"  Accuracy         : {acc_claim:.4f}")
    print(f"  Weighted F1      : {f1_claim:.4f}")
    print(f"  Rejected Recall  : {rejected_recall:.4f}")
    print(f"  Run ID           : {claim_run_id}")

2026/07/22 15:07:40 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


RandomForest Claim Model logged ✓
  Accuracy         : 0.4936
  Weighted F1      : 0.4870
  Rejected Recall  : 0.3375
  Run ID           : 8f799153757e42e79b18dd41ef1f6553


In [28]:
print("=" * 60)
print("MLFLOW RUNS SUMMARY")
print("=" * 60)
print(f"Risk  Model Run ID : {risk_run_id}")
print(f"Claim Model Run ID : {claim_run_id}")
print()
print("Open MLflow UI at: http://127.0.0.1:5000")

MLFLOW RUNS SUMMARY
Risk  Model Run ID : 5ce2933f9346427388a5b210020a9396
Claim Model Run ID : 8f799153757e42e79b18dd41ef1f6553

Open MLflow UI at: http://127.0.0.1:5000


# Register the Model
Registration creates a named, versioned entry in the MLflow Model Registry.

Every time you register the same name, the version number increments automatically. v1 → v2 → v3 and so on.

In [29]:
from mlflow import register_model

register_model_name = "HealthcareRiskRFModel"
model_uri = f"runs:/{risk_run_id}/model"
result = register_model(
    model_uri = model_uri,
    name = register_model_name
)

print(f"Register Model Name: ", result.name)
print(f"Register Model Version: ", result.version)

Successfully registered model 'HealthcareRiskRFModel'.
2026/07/22 19:42:09 WARNING mlflow.tracking._model_registry.fluent: Run with id 5ce2933f9346427388a5b210020a9396 has no artifacts at artifact path 'model', registering model based on models:/m-d7f43e60357a446f9a0679994d76030d instead


Register Model Name:  HealthcareRiskRFModel
Register Model Version:  1


Created version '1' of model 'HealthcareRiskRFModel'.


In [30]:
# Saving the registered version in a variable
risk_model_version = result.version
print(f"Risk model version saved: ", risk_model_version)

Risk model version saved:  1


In [31]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
print("Mlflow Client ready ✓")

Mlflow Client ready ✓


In [32]:
# Move Risk Model to production
client.transition_model_version_stage(
    name = register_model_name,
    version = risk_model_version,
    stage = "Staging"
)

print(f"Model {register_model_name} version {risk_model_version} moved to staging")

Model HealthcareRiskRFModel version 1 moved to staging


In [33]:
# Check if model qualifies for production
if acc_risk >= 0.55 and high_recall >= 0.70:
    print("Risk Model is eligible to be promoted to Production")
else:
    print("Risk Model is not eligible to be promoted to Production")

Risk Model is eligible to be promoted to Production


In [34]:
# Promote Risk model to production

client.transition_model_version_stage(
    name = register_model_name,
    version = risk_model_version,
    stage = "Production",
    archive_existing_versions = True
)

print(f"Model {register_model_name} version {risk_model_version} moved to production")

Model HealthcareRiskRFModel version 1 moved to production


In [36]:
# Load production Risk Model from registry
import mlflow.sklearn

production_risk_model = mlflow.sklearn.load_model(
    model_uri = f"models:/{register_model_name}/Production"
)

print(f"Production Risk Model Loaded")

Production Risk Model Loaded


In [39]:
# Fetch current production version

latest_versions = client.get_latest_versions(
    name = register_model_name,
    stages = ["Production"]
)

production_version = latest_versions[0].version if latest_versions else None
print("Current Production Version: ",production_version)

Current Production Version:  1


In [40]:
# Run a prediction using Production Risk Model
pred_risk = production_risk_model.predict(X_test_risk.head(5))
print("Prediction: ", pred_risk)

Prediction:  ['Medium' 'Medium' 'Medium' 'Low' 'Medium']


In [41]:
# Log prediction with model version
import hashlib
from datetime import datetime

def hash_input(payload):
    payload_str = json.dumps(payload, sort_keys=True)
    return hashlib.sha256(payload_str.encode()).hexdigest()

In [42]:
# Input Payload

input_payload = {
    "age": 52,
    "gender": "M",                        
    "city": "Bangalore",
    "insurance_provider": "CareOne",       
    "chronic_flag": 1,
    "department": "Cardiology",
    "visit_type": "ER",                    
    "doctor_id": 101,
    "length_of_stay_hours": 48,
    "days_since_registration": 300,
    "visit_frequency": 4,
    "avg_los_per_patient": 36.5,
    "visit_month": 3,
    "visit_dayofweek": 2
}

input_hash = hash_input(input_payload)
print("Input Hash: ", input_hash)

Input Hash:  393af61ee47c8f37fa64b93931d86baa4a2690fc9a9ecf27987b660d34e5c8bd


In [44]:
# Create a Log Record

BASE_DIR = os.getcwd()
LOG_DIR = os.path.join(BASE_DIR, "logs")

os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = os.path.join(LOG_DIR, "predictions.log")

prediction_log = {
    "timestamp": datetime.utcnow().isoformat(),
    "model_name": register_model_name,
    "model_version": production_version,
    "input_hash": input_hash,
    "prediction": str(pred_risk[0])
}

with open(LOG_FILE, "a", encoding="utf-8") as f:
    f.write(json.dumps(prediction_log) + "\n")

print("Prediction logged with model_version ✓")


Prediction logged with model_version ✓
